# NB06 — Export for *Translate*

**Role in the pipeline.** This notebook is a **pure serializer**. It does *not*
fit any model or recompute any science: it reads the numeric hand-off artifacts
produced upstream (NB03 PCA, NB04 Isolation Forest, NB05 autoencoder) and the
Copernicus GREP ensemble, and writes a single **compact static JSON bundle** for
the *Translate* phase — a Three.js + D3 front-end living in the separate repo
`amoc-thesis-translate`.

**Design contract**

| Requirement | How it is met |
|---|---|
| Compact, `< 500 KB` total | monthly-resampled trajectory + strided manifold cloud + aggressive rounding + minified JSON; a hard `assert` guards the budget |
| Documented schema | every block is described in markdown here, and the notebook writes a `README.md` next to the JSON |
| Preserve PC1–PC2 in time | `trajectory_monthly` (ordered path) + `manifold_cloud` (distribution) |
| Anomaly-cluster centroids | `anomalies.clusters[*].centroid_pc1/pc2`, from NB04's flags |
| Copernicus ensemble uncertainty | `copernicus_ensemble` mean / std / member envelope |
| Reproducible, deterministic, **no random state** | this notebook contains **no stochastic step**; the one seeded step (Isolation Forest, `random_state=42`) lives upstream in NB04 and is consumed here as a static `.npz`. Re-running this notebook is byte-identical. |

**Inputs** (all under `03_Data/processed/`, via `pipeline.paths`)
- `pca_vertical_scores.npz` — NB03: `pc1, pc2, eof1, eof2, mean_profile, depth, var_explained, times`
- `ae_latent_scores.npz` — NB05: `ae1, ae2, recon_err, r2_by_k, active_mask, times`
- `anomaly_flags.npz` — NB04 Isolation Forest: `is_anomaly, anomaly_score, times` (+ params)
- Copernicus GREP ensemble — via `pipeline.data_loading.load_copernicus` (already in Sv)

**Output**
- `03_Data/outputs/translate/amoc_translate_bundle.json`
- `03_Data/outputs/translate/README.md` (schema, generated from this notebook)


## 1 · Imports & configuration

Stack is intentionally minimal: `numpy`, `pandas`, the standard-library `json`,
plus the project's own `pipeline` package (which loads the Copernicus netCDF and
converts it to Sverdrups — the single source of truth for data loading). No other
third-party dependency is used here.

All tunables live in one block so the export is easy to audit and re-run.

In [11]:
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# --- Make the project `pipeline` package importable (notebooks/ -> 04_Code/) ---
NOTEBOOK_DIR = Path.cwd()
PROJECT_CODE = NOTEBOOK_DIR.parent
if str(PROJECT_CODE) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE))

from pipeline.paths import DATA_PROCESSED, DATA_OUTPUTS
from pipeline.data_loading import load_copernicus

# ============================================================
#  CONFIGURATION  (every tunable in one place)
# ============================================================
SCHEMA_VERSION = "1.0.0"

# Output location: the pipeline's designated outputs dir. Copy this folder into
# `amoc-thesis-translate/public/data/` (or equivalent) for the front-end.
OUT_DIR   = DATA_OUTPUTS / "translate"
OUT_JSON  = OUT_DIR / "amoc_translate_bundle.json"
OUT_README = OUT_DIR / "README.md"
SIZE_BUDGET_BYTES = 500 * 1024          # hard 500 KB ceiling

# Trajectory: daily RAPID (14,596 pts) is too dense for a public 20-year view.
# Resample to month-start means — smooth, ordered, and tiny.
RESAMPLE = "MS"                         # month-start

# Manifold point cloud: keep the full-range distribution to show the flat/thin
# manifold, but stride it so the cloud stays compact.
CLOUD_STRIDE_DAYS = 5                   # every 5th day (~2,920 pts)

# Anomaly temporal clustering (over NB04's Isolation-Forest flags).
# GAP_DAYS: max gap between flagged days inside one cluster.
# MIN_CLUSTER_DAYS: minimum flagged days for a cluster to be named/exported.
GAP_DAYS = 60
MIN_CLUSTER_DAYS = 15

# Rounding budget (decimals) per channel — the main size lever.
ND_PC     = 2      # PC scores, range ~ +/-286
ND_AE     = 4      # AE latent, range ~ +/-1.3
ND_ERR    = 4      # reconstruction RMSE (Sv), range ~ 0.07-2.27
ND_SV     = 3      # Copernicus AMOC (Sv), range ~ 10-25
ND_EOF    = 6      # EOF mode shapes (small, normalized)
ND_DEPTH  = 1      # depth (m)
ND_VAR    = 5      # variance-explained fractions

print("Config loaded. Output ->", OUT_JSON)


Config loaded. Output -> C:\Users\Vaccari\Desktop\iCloudDrive\Desktop\ENRICO\05_LEARNING\University\ToU\Phases\04_Activation_Phase\03_Data\outputs\translate\amoc_translate_bundle.json


## 2 · Load the upstream artifacts

No computation here beyond reading `.npz` files and the Copernicus dataset. We
assert the three daily series share an identical time axis (they are all built on
the same RAPID daily grid), so downstream indexing is safe.

In [12]:
# --- NB03: PCA on vertical profiles ---
pca = np.load(DATA_PROCESSED / "pca_vertical_scores.npz", allow_pickle=True)
pc1, pc2      = pca["pc1"], pca["pc2"]
eof1, eof2    = pca["eof1"], pca["eof2"]
mean_profile  = pca["mean_profile"]           # climatological mean, Sv (NB03 owns it)
depth         = pca["depth"]
var_explained = pca["var_explained"]
times_pca     = pca["times"]

# --- NB05: autoencoder latent + reconstruction error ---
ae = np.load(DATA_PROCESSED / "ae_latent_scores.npz", allow_pickle=True)
ae1, ae2    = ae["ae1"], ae["ae2"]
recon_err   = ae["recon_err"]
r2_by_k     = ae["r2_by_k"]
active_mask = ae["active_mask"]
times_ae    = ae["times"]

# --- NB04: Isolation-Forest anomaly flags (seeded upstream, static here) ---
anom = np.load(DATA_PROCESSED / "anomaly_flags.npz", allow_pickle=True)
is_anomaly    = anom["is_anomaly"].astype(bool)
anomaly_score = anom["anomaly_score"]
times_anom    = anom["times"]
if_params = {
    "contamination": float(anom["contamination"]),
    "n_estimators":  int(anom["n_estimators"]),
    "random_state":  int(anom["random_state"]),
}

# --- Shared daily time axis (RAPID grid) ---
assert (times_pca == times_ae).all() and (times_pca == times_anom).all(), \
    "PCA / AE / anomaly time axes must be identical"
t_daily = pd.to_datetime(times_pca)
n_days  = len(t_daily)

# --- Copernicus GREP ensemble (monthly, already Sv via the loader) ---
cop = load_copernicus()
cop_members = ["amoc_cglo", "amoc_glor", "amoc_oras"]

assert mean_profile.shape == depth.shape, "mean_profile must align with depth axis"
print(f"mean_profile       : {mean_profile.shape}  "
      f"(range {mean_profile.min():.1f} .. {mean_profile.max():.1f} Sv)")
print(f"RAPID daily series : {n_days:,} days  "
      f"({t_daily[0].date()} -> {t_daily[-1].date()})")
print(f"Copernicus ensemble: {cop.time.size} months "
      f"({str(cop.time.values[0])[:7]} -> {str(cop.time.values[-1])[:7]})")
print(f"Isolation Forest   : {int(is_anomaly.sum())} flagged days "
      f"({100*is_anomaly.mean():.2f}%), params={if_params}")


mean_profile       : (307,)  (range -1.1 .. 16.8 Sv)
RAPID daily series : 14,596 days  (2004-04-02 -> 2024-03-25)
Copernicus ensemble: 372 months (1993-01 -> 2023-12)
Isolation Forest   : 730 flagged days (5.00%), params={'contamination': 0.05, 'n_estimators': 200, 'random_state': 42}


## 3 · Serialization helpers

Two rules keep the bundle small and valid:
1. **Round** every float to the minimum decimals a public visual needs.
2. Convert `NaN`/`inf` to JSON `null` (front-end must skip nulls).

`clean()` recurses the assembled bundle to enforce native Python types and null
handling right before writing.

In [13]:
def rnd(x, nd):
    """Round a scalar to `nd` decimals -> float, or None for non-finite."""
    x = float(x)
    return None if not np.isfinite(x) else round(x, nd)

def rnd_list(arr, nd):
    """Round a 1-D array -> list[float|None], compact and JSON-safe."""
    a = np.asarray(arr, dtype=float)
    return [None if not np.isfinite(v) else round(float(v), nd) for v in a]

def month_labels(idx):
    """DatetimeIndex -> ['YYYY-MM', ...] (month-start labels)."""
    return [d.strftime("%Y-%m") for d in idx]

def clean(o):
    """Recursively coerce numpy types -> native, non-finite floats -> None."""
    if isinstance(o, dict):
        return {k: clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [clean(v) for v in o]
    if isinstance(o, (np.floating, float)):
        f = float(o)
        return f if np.isfinite(f) else None
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_, bool)):
        return bool(o)
    return o

print("Helpers ready.")


Helpers ready.


## 4 · Block: `pca`

The vertical-profile PCA basis. PC1 (~90 %) is AMOC **intensity**; PC2 (~8 %) is
vertical **redistribution**; together **97.86 %**.

**Reconstruction (Scene II).** A profile is rebuilt as
`mean_profile + pc1·recon_basis.pc1 + pc2·recon_basis.pc2` (Sv). Two subtleties
made this fail before, both fixed here:
1. **The mean was missing.** PCA centering removed `mean_profile`, so it is not in
   the npz — we rebuild it from RAPID (§2).
2. **`eof1/eof2` are Sv-scaled loadings** (`components·√λ`), meant for *display*
   of the mode shape — **not** a reconstruction basis for the raw scores. Using
   them directly overshoots by `√λ` (~10×). The reconstruction basis is the
   **unit-length** vector, recovered as `eof / std(pc)` (since `std(pc)=√λ`).

`mean_profile` comes straight from NB03's npz (the authoritative `pca.mean_`);
a hard assertion checks the derived basis vectors are unit-length, so the
`eof → basis` rescaling can't silently drift.

In [14]:
# Unit reconstruction basis: std(pc, ddof=1) == sqrt(eigenvalue), so eof/std
# recovers the unit-length component that pairs with the RAW scores.
scale1, scale2 = pc1.std(ddof=1), pc2.std(ddof=1)
basis1, basis2 = eof1 / scale1, eof2 / scale2

# Self-validation: the reconstruction basis must be unit-length (that is exactly
# what makes `mean + pc*basis` reproduce sklearn's `mean + score*component`).
n1, n2 = float(np.linalg.norm(basis1)), float(np.linalg.norm(basis2))
assert abs(n1 - 1) < 1e-6 and abs(n2 - 1) < 1e-6, \
    f"recon basis not unit-length (|b1|={n1:.6f}, |b2|={n2:.6f})"
print(f"reconstruction basis self-check: |basis_pc1|={n1:.6f}, |basis_pc2|={n2:.6f} (unit)")

pca_block = {
    "var_explained": rnd_list(var_explained[:5], ND_VAR),
    "cumulative_2pc": rnd(float(var_explained[:2].sum()), ND_VAR),
    "n_levels": int(depth.size),
    "depth_m": rnd_list(depth, ND_DEPTH),
    "mean_profile": rnd_list(mean_profile, ND_SV),     # Psi-bar(z), Sv  (NEW)
    "eof1": rnd_list(eof1, ND_EOF),                    # Sv-scaled loading = mode SHAPE (display)
    "eof2": rnd_list(eof2, ND_EOF),
    "recon_basis": {                                   # unit vectors for reconstruction
        "pc1": rnd_list(basis1, ND_EOF),
        "pc2": rnd_list(basis2, ND_EOF),
    },
    "reconstruction":
        "profile(t) = mean_profile + pc1(t)*recon_basis.pc1 + pc2(t)*recon_basis.pc2  (Sv)",
    "notes": {
        "pc1": "AMOC intensity (dominant mode)",
        "pc2": "vertical redistribution (second mode)",
        "eof_vs_basis": "eof1/eof2 are Sv-scaled loadings for DISPLAY (mode shape); "
                        "use recon_basis (unit vectors) with the raw pc1/pc2 to rebuild profiles.",
    },
}
print("pca block:", {k: (len(v) if isinstance(v, list) else v)
                      for k, v in pca_block.items() if k not in ('notes', 'recon_basis')})


reconstruction basis self-check: |basis_pc1|=1.000000, |basis_pc2|=1.000000 (unit)
pca block: {'var_explained': 5, 'cumulative_2pc': 0.97855, 'n_levels': 307, 'depth_m': 307, 'mean_profile': 307, 'eof1': 307, 'eof2': 307, 'reconstruction': 'profile(t) = mean_profile + pc1(t)*recon_basis.pc1 + pc2(t)*recon_basis.pc2  (Sv)'}


## 5 · Blocks: `trajectory_monthly` & `manifold_cloud`

Two complementary views of the same daily series:

- **`trajectory_monthly`** — month-start means of `pc1, pc2, ae1, ae2, recon_err`.
  An ordered path for the animated time-walk through the latent plane (~240 pts).
- **`manifold_cloud`** — the daily record strided every `CLOUD_STRIDE_DAYS` days,
  as `(pc1, pc2)` and `(ae1, ae2)` point clouds. This is what makes the **flat /
  thin manifold** visually obvious (a 2-D cloud collapsing onto ~1 dimension).

The full daily arrays remain in the `.npz` files for any higher-fidelity need.

In [15]:
# --- Monthly trajectory (ordered path) ---
df = pd.DataFrame(
    {"pc1": pc1, "pc2": pc2, "ae1": ae1, "ae2": ae2, "recon_err": recon_err},
    index=t_daily,
)
mon = df.resample(RESAMPLE).mean()
trajectory_monthly = {
    "resample": RESAMPLE,
    "time": month_labels(mon.index),
    "pc1": rnd_list(mon["pc1"], ND_PC),
    "pc2": rnd_list(mon["pc2"], ND_PC),
    "ae1": rnd_list(mon["ae1"], ND_AE),
    "ae2": rnd_list(mon["ae2"], ND_AE),
    "recon_err": rnd_list(mon["recon_err"], ND_ERR),
}

# --- Strided manifold cloud (distribution / flatness) ---
sl = slice(None, None, CLOUD_STRIDE_DAYS)
manifold_cloud = {
    "stride_days": CLOUD_STRIDE_DAYS,
    "n_points": int(np.arange(n_days)[sl].size),
    "pc": {"pc1": rnd_list(pc1[sl], ND_PC), "pc2": rnd_list(pc2[sl], ND_PC)},
    "ae": {"ae1": rnd_list(ae1[sl], ND_AE), "ae2": rnd_list(ae2[sl], ND_AE)},
}
print(f"trajectory_monthly: {len(trajectory_monthly['time'])} months")
print(f"manifold_cloud    : {manifold_cloud['n_points']} points "
      f"(stride {CLOUD_STRIDE_DAYS}d)")


trajectory_monthly: 240 months
manifold_cloud    : 2920 points (stride 5d)


## 6 · Block: `autoencoder`

The k-sweep result: R² (%) of the reconstruction as a function of latent
dimensionality k = 1..5, measured on all 307 levels in Sv. The jump from k=1 to
k=2 and the near-flat tail beyond k=2 is the quantitative statement that **two
numbers already explain the profile** — the manifold is essentially 2-D and flat,
which the front-end narrates alongside the PCA's 97.86 %.

In [16]:
autoencoder_block = {
    "r2_by_k_percent": rnd_list(r2_by_k, 4),          # k = 1..5
    "k_values": [1, 2, 3, 4, 5],
    "n_active_levels": int(active_mask.sum()),        # levels the AE saw
    "architecture": "305 -> 128 -> 32 -> k -> 32 -> 128 -> 305 (Tanh, k=2 shipped)",
    "note": "Non-linear AE barely beats linear PCA at equal k -> manifold is flat.",
}
print("autoencoder block:", autoencoder_block["r2_by_k_percent"])


autoencoder block: [89.673, 97.7718, 99.3591, 99.7363, 99.8338]


## 7 · Block: `anomalies`

NB04 flags 5 % of days as anomalous with an Isolation Forest on `(PC1, PC2)`
(`random_state=42`, consumed here as a static `.npz` — **no model is fit in this
notebook**). We turn the flagged days into **temporal clusters**: flagged days
within `GAP_DAYS` of each other form one cluster; clusters with `>= MIN_CLUSTER_DAYS`
days are named `YYYY` or `YYYY-YYYY` and given a **centroid** in the PC plane.

The dominant cluster is the documented **2009–2010** AMOC weakening (strongly
negative PC1). We also ship the flagged points themselves (strided) so the
front-end can highlight them over the manifold cloud.

In [17]:
# --- Group consecutive flagged days into temporal clusters ---
flag_idx = np.where(is_anomaly)[0]
groups, cur = [], [flag_idx[0]]
for a, b in zip(flag_idx[:-1], flag_idx[1:]):
    if (t_daily[b] - t_daily[a]).days <= GAP_DAYS:
        cur.append(b)
    else:
        groups.append(cur); cur = [b]
groups.append(cur)

clusters = []
for g in groups:
    if len(g) < MIN_CLUSTER_DAYS:
        continue
    gi = np.asarray(g)
    y0, y1 = t_daily[gi[0]].year, t_daily[gi[-1]].year
    clusters.append({
        "label": f"{y0}" if y0 == y1 else f"{y0}-{y1}",
        "t_start": str(t_daily[gi[0]].date()),
        "t_end":   str(t_daily[gi[-1]].date()),
        "n_days":  int(gi.size),
        "centroid_pc1": rnd(pc1[gi].mean(), ND_PC),
        "centroid_pc2": rnd(pc2[gi].mean(), ND_PC),
        "mean_anomaly_score": rnd(anomaly_score[gi].mean(), 4),
    })
clusters.sort(key=lambda c: c["n_days"], reverse=True)
for rank, c in enumerate(clusters):        # rank 0 = largest cluster
    c["rank"] = rank

# --- Flagged points (strided) for highlighting over the cloud ---
fsl = flag_idx[::2]                          # ~365 pts, plenty for a scatter
anomalies_block = {
    "method": "IsolationForest on (PC1, PC2) — NB04",
    "params": if_params,
    "n_flagged": int(is_anomaly.sum()),
    "pct_flagged": rnd(100 * is_anomaly.mean(), 2),
    "clustering": {"gap_days": GAP_DAYS, "min_cluster_days": MIN_CLUSTER_DAYS},
    "clusters": clusters,
    "flagged_points": {
        "pc1": rnd_list(pc1[fsl], ND_PC),
        "pc2": rnd_list(pc2[fsl], ND_PC),
    },
}
print(f"{len(clusters)} clusters; top: "
      + ", ".join(f"{c['label']}(n={c['n_days']})" for c in clusters[:4]))
print("primary centroid:", clusters[0]["label"],
      (clusters[0]["centroid_pc1"], clusters[0]["centroid_pc2"]))


18 clusters; top: 2009-2010(n=85), 2010-2011(n=67), 2017-2018(n=60), 2005-2006(n=37)
primary centroid: 2009-2010 (-158.64, 2.35)


## 8 · Block: `copernicus_ensemble`

The Copernicus GREP reanalysis ensemble (three members: C-GLORS, GLORYS,
ORAS5) at 26 °N, monthly, 1993–2023, in Sv. We ship the **uncertainty band** two
compatible ways so the front-end can choose:
- `mean` ± `std` (Gaussian band), and
- `lower`/`upper` = the **member envelope** (min/max across the 3 members) — the
  honest "spread of the models" band used in the thesis's over-confidence point.

Individual `members` are included (tiny) for spaghetti plots.

In [18]:
cop_df = cop[cop_members].to_dataframe()[cop_members]
members_arr = cop_df.to_numpy()                       # (372, 3)
env_lower = members_arr.min(axis=1)
env_upper = members_arr.max(axis=1)
cop_time  = pd.to_datetime(cop["time"].values)

copernicus_block = {
    "units": "Sv",
    "source": "Copernicus GREP (C-GLORS, GLORYS, ORAS5) @ 26N",
    "time": month_labels(cop_time),
    "mean": rnd_list(cop["amoc_mean"].values, ND_SV),
    "std":  rnd_list(cop["amoc_std"].values, ND_SV),
    "lower": rnd_list(env_lower, ND_SV),              # member envelope min
    "upper": rnd_list(env_upper, ND_SV),              # member envelope max
    "members": {
        "cglo": rnd_list(cop["amoc_cglo"].values, ND_SV),
        "glor": rnd_list(cop["amoc_glor"].values, ND_SV),
        "oras": rnd_list(cop["amoc_oras"].values, ND_SV),
    },
    "band_definition": {
        "gaussian": "mean +/- std",
        "envelope": "lower/upper = min/max across the 3 members",
    },
}
print(f"copernicus_ensemble: {len(copernicus_block['time'])} months, "
      f"mean {np.nanmean(cop['amoc_mean'].values):.1f} Sv")


copernicus_ensemble: 372 months, mean 17.2 Sv


## 9 · Assemble, serialize, and enforce the size budget

The bundle is minified (`separators=(",", ":")`, `ensure_ascii=False`) after
`clean()`. A `sha256` of the bytes is embedded in `meta` for provenance, and a
hard `assert` fails the run if the file exceeds 500 KB.

In [19]:
meta = {
    "schema_version": SCHEMA_VERSION,
    "title": "AMOC ML representations — Translate bundle",
    "description": "Compact static export of NB03 (PCA), NB04 (Isolation Forest) "
                   "and NB05 (autoencoder) results for the Three.js + D3 front-end.",
    "generated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "units": {"pc": "Sv-projection", "amoc": "Sv", "depth": "m",
              "recon_err": "Sv (RMSE)"},
    "time_axes": {
        "rapid_daily_span": [str(t_daily[0].date()), str(t_daily[-1].date())],
        "trajectory_resample": RESAMPLE,
        "copernicus_span": [copernicus_block["time"][0], copernicus_block["time"][-1]],
    },
    "provenance": {
        "source_notebooks": {
            "pca": "03_PCA_on_vertical_profiles.ipynb",
            "anomalies": "04_anomalies_early_warning_signals.ipynb",
            "autoencoder": "05_autoencoder_vertical_profiles.ipynb",
            "export": "06_export_for_translate.ipynb",
        },
        "inputs": ["pca_vertical_scores.npz", "ae_latent_scores.npz",
                   "anomaly_flags.npz", "Copernicus GREP (via pipeline loader)"],
    },
    "determinism": "No stochastic step in this notebook; the only seeded step "
                   "(Isolation Forest, random_state=42) is upstream in NB04. "
                   "meta.data_sha256 fingerprints the scientific payload and is "
                   "stable across runs; only generated_utc varies.",
}

bundle = clean({
    "meta": meta,
    "pca": pca_block,
    "trajectory_monthly": trajectory_monthly,
    "manifold_cloud": manifold_cloud,
    "autoencoder": autoencoder_block,
    "anomalies": anomalies_block,
    "copernicus_ensemble": copernicus_block,
})

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Stable fingerprint of the SCIENTIFIC payload only (excludes volatile meta, so
# the hash is identical across runs and serves as the determinism check).
sci = {k: v for k, v in bundle.items() if k != "meta"}
sci_payload = json.dumps(sci, separators=(",", ":"), ensure_ascii=False, sort_keys=True)
bundle["meta"]["data_sha256"] = hashlib.sha256(sci_payload.encode("utf-8")).hexdigest()

final = json.dumps(bundle, separators=(",", ":"), ensure_ascii=False)
OUT_JSON.write_text(final, encoding="utf-8")

size = OUT_JSON.stat().st_size
print(f"Wrote {OUT_JSON}")
print(f"Size : {size:,} bytes  ({size/1024:.1f} KB)  "
      f"[{100*size/SIZE_BUDGET_BYTES:.1f}% of 500 KB budget]")
assert size <= SIZE_BUDGET_BYTES, f"Bundle {size} B exceeds 500 KB budget"

# Per-block size breakdown (helps future tuning).
print("\nApprox. size per top-level block:")
for k, v in bundle.items():
    b = len(json.dumps(v, separators=(",", ":"), ensure_ascii=False).encode())
    print(f"  {k:20s} {b/1024:6.1f} KB")


Wrote C:\Users\Vaccari\Desktop\iCloudDrive\Desktop\ENRICO\05_LEARNING\University\ToU\Phases\04_Activation_Phase\03_Data\outputs\translate\amoc_translate_bundle.json
Size : 136,363 bytes  (133.2 KB)  [26.6% of 500 KB budget]

Approx. size per top-level block:
  meta                    1.1 KB
  pca                    15.6 KB
  trajectory_monthly     10.3 KB
  manifold_cloud         77.1 KB
  autoencoder             0.3 KB
  anomalies               7.8 KB
  copernicus_ensemble    21.0 KB


## 10 · Write the schema `README.md`

The export folder is self-describing: a `README.md` documenting every key ships
next to the JSON, so the `amoc-thesis-translate` team can consume the bundle
without reading this notebook.

In [20]:
readme = f"""# AMOC → Translate data bundle

`amoc_translate_bundle.json` — schema **v{SCHEMA_VERSION}**. Static, compact
({size/1024:.0f} KB), read-only. Generated by `04_Code/notebooks/06_export_for_translate.ipynb`.
All AMOC quantities are in **Sverdrups (Sv)**; depths in **metres**. Non-finite
values are serialized as `null` — skip them.

## Top-level keys

| key | what it is |
|---|---|
| `meta` | schema version, provenance, time axes, sha256, determinism note |
| `pca` | PCA basis: variance explained, `depth_m`, `mean_profile`, `eof1/eof2` (Sv display loadings), `recon_basis` (unit vectors) |
| `trajectory_monthly` | month-start means of `pc1,pc2,ae1,ae2,recon_err` (ordered path) |
| `manifold_cloud` | daily record strided {CLOUD_STRIDE_DAYS}d as `(pc1,pc2)` & `(ae1,ae2)` clouds |
| `autoencoder` | R² vs latent-k sweep, architecture, flatness note |
| `anomalies` | Isolation-Forest clusters with PC-plane `centroid_pc1/pc2` + flagged points |
| `copernicus_ensemble` | GREP monthly mean/std + member envelope (`lower/upper`) + members |

## Field notes

- **Parallel arrays.** Within a block, arrays are index-aligned (e.g.
  `trajectory_monthly.time[i]` ↔ `.pc1[i]`). `time` labels are `"YYYY-MM"`.
- **Reconstruct a vertical profile (Scene II):**
  `profile ≈ mean_profile + pc1·recon_basis.pc1 + pc2·recon_basis.pc2` over
  `pca.depth_m` (Sv). Use the **unit** `recon_basis` vectors with the raw
  `pc1/pc2` — **not** `eof1/eof2`, which are Sv-scaled *display* loadings
  (mode shape) and would overshoot by ~10×. All arrays align to `depth_m`.
- **PC plane:** PC1 ≈ AMOC intensity, PC2 ≈ vertical redistribution.
  `manifold_cloud` makes the ~1-D flatness visible.
- **Anomalies:** `anomalies.clusters` is sorted by size (`rank` 0 = largest);
  the dominant `2009-2010` cluster sits at strongly negative PC1 (weak AMOC).
  `flagged_points` are strided flagged days for highlighting.
- **Copernicus band:** use `mean ± std` (Gaussian) or `lower/upper` (min/max
  across the three members) — the honest "spread of models" envelope.

## Regenerating

```
conda activate amoc-thesis
jupyter nbconvert --to notebook --execute \\
  notebooks/06_export_for_translate.ipynb --output 06_export_for_translate.ipynb
```

Deterministic: the only seeded step (Isolation Forest, `random_state=42`) is
upstream in NB04; this export contains no randomness. Re-running yields an
identical scientific payload — verify via the stable `meta.data_sha256`
(only `meta.generated_utc` changes between runs).
"""
OUT_README.write_text(readme, encoding="utf-8")
print("Wrote", OUT_README)


Wrote C:\Users\Vaccari\Desktop\iCloudDrive\Desktop\ENRICO\05_LEARNING\University\ToU\Phases\04_Activation_Phase\03_Data\outputs\translate\README.md


---
**Done.** Copy `03_Data/outputs/translate/` into the `amoc-thesis-translate`
repo (e.g. `public/data/`). Re-running top-to-bottom reproduces an identical
scientific payload (stable `meta.data_sha256`); only `meta.generated_utc` differs.